# GPLFR quickstart

Fit a GPLFR model on a small synthetic toy problem. This notebook shows the
canonical `fit -> predict -> plot` flow using `gplfr.applications.toy.GPLFR`.

For the full toy experiments behind the paper figures, see `applications/toy/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from gplfr.applications.toy.gplfr import GPLFR

torch.manual_seed(0)
rng = np.random.default_rng(0)

## Synthetic data

`N=30` training points in `D_x=2`, outputs on a `4x4` grid. The toy GPLFR
implementation fits flattened outputs, so we keep both a gridded view for
plotting and a flattened view for training.

In [ ]:
N, D_x, H, W, K = 30, 2, 4, 4, 2
X = rng.standard_normal((N, D_x)).astype(np.float64)
latents = rng.standard_normal((N, K)).astype(np.float64)
W_dec = rng.standard_normal((K, H * W)).astype(np.float64) * 0.5
Y_flat = latents @ W_dec + rng.standard_normal((N, H * W)).astype(np.float64) * 0.1

X_t = torch.from_numpy(X)
Y_t = torch.from_numpy(Y_flat)
print(f"X: {tuple(X_t.shape)}  Y: {tuple(Y_t.shape)}")

## Fit GPLFR

We match the generative rank with `latent_dim=2` and run a short MAP fit via
SVI. This is meant to stay lightweight rather than paper-grade.

In [ ]:
model = GPLFR(latent_dim=2, kernel="matern52", ell_mode="shared", dtype=torch.float32)
fit_result = model.fit(
    X_t,
    Y_t,
    num_steps=60,
    lr_Z=5.0e-2,
    lr_global=1.0e-2,
    log_every=30,
    verbose=False,
)
print(f"final ELBO loss: {fit_result.final_loss:.4f}")

## Predict on a held-out grid

In [ ]:
X_test = rng.standard_normal((8, D_x)).astype(np.float64)
Y_pred = model.predict(torch.from_numpy(X_test)).reshape(-1, H, W)
print(f"predictions shape: {Y_pred.shape}  (n_test, H, W)")

## Inspect a prediction

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
im0 = ax[0].imshow(Y_pred[0], cmap="viridis")
ax[0].set_title("prediction[0]")
fig.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(Y_pred[1], cmap="viridis")
ax[1].set_title("prediction[1]")
fig.colorbar(im1, ax=ax[1])
plt.show()

## Next steps

- Paper toy experiments: `applications/toy/`
- PyXOpto reflectance experiments: `applications/pyxopto/`
- ExoWorldsBench climate-emulation entrypoints: `applications/exoclimate/`